# Graph-Paper AI — Colab Test

Vectorless Graph-RAG: PDF → Graph → Tree Search (LLM) → Answer.

Works with or without a local Ollama instance.

## Setup

In [2]:
# Clone the repo and install deps
import sys, subprocess, os

REPO_URL = "https://github.com/ahmadoubg/graph-paper-ai.git"
REPO_DIR = "/content/graph-paper-ai"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q pymupdf networkx httpx llama-cloud

Cloning into '/content/graph-paper-ai'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 173 (delta 73), reused 133 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 130.65 KiB | 5.68 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/graph-paper-ai
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 20.0 MB/s eta 0:00:00


In [3]:
import os

# Get your free API key from https://cloud.llamaindex.ai
os.environ["LLAMACLOUD_API_KEY"] = "YOUR_LLAMACLOUD_API_KEY"  # <--- REPLACE WITH YOUR KEY

In [4]:
from pathlib import Path
from pprint import pprint

import networkx as nx

from src.ingestion import (
    build_adjacency_map, build_section_tree, extract_cross_references, print_tree,
)
from src.retrieval.tree_search import answer_query, tree_search

## 1. Get a PDF

Run this cell, then click the **Upload** button to select a PDF from your computer.

In [5]:
from google.colab import files

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
PDF_PATH = Path(pdf_name)
OUTPUT_DIR = Path("output")
print(f"Using: {PDF_PATH} ({PDF_PATH.stat().st_size / 1024:.0f} KB)")

Saving paper5.pdf to paper5.pdf
Using: paper5.pdf (1099 KB)


In [6]:
from google.colab import userdata
key_api = userdata.get('LLAMACLOUD_API_KEY')

### Or download a sample paper automatically

In [ ]:
# Uncomment to skip upload and use a sample arXiv paper instead:
# PDF_URL = "https://arxiv.org/pdf/2401.12345.pdf"  # Change this
# !wget -q -O paper.pdf {PDF_URL}
# PDF_PATH = Path("paper.pdf")
# OUTPUT_DIR = Path("output")
# print(f"Downloaded: {PDF_PATH}")

## 2. Parse the PDF

Extracts markdown (with `## Page N` markers), images, and spatial co-location edges.

In [17]:
from llama_cloud import AsyncLlamaCloud
from google.colab import userdata
from src.ingestion import ProcessingResult

async def parse_with_llamaparse(pdf_path):
    key_api = userdata.get('LLAMACLOUD_API_KEY')
    client = AsyncLlamaCloud(api_key=key_api)

    file_obj = await client.files.create(file=pdf_path, purpose="parse")

    # Request only markdown_full, as markdown_sections is not a valid expand parameter
    llamaparse_result = await client.parsing.parse(
        file_id=file_obj.id,
        tier="agentic",
        expand=["markdown_full"],
        version='2026-06-04'
    )
    return llamaparse_result

# Run the async function
llamaparse_raw_result = await parse_with_llamaparse(PDF_PATH)
llamaparse_markdown = llamaparse_raw_result.markdown_full if llamaparse_raw_result else ""
# Since markdown_sections is not available, we will not try to extract it
llamaparse_sections = []

print(f"\nSuccess! LlamaParse generated {len(llamaparse_markdown):,} characters.")
print("\n--- LlamaParse Markdown Preview (First 1000 chars) ---")
print(llamaparse_markdown[:1000])

# Adapt the output of llamaparse to fit the ProcessingResult structure.
# Dummy image and edge lists for now.

dummy_images = []
dummy_edges = []

# Infer page_count from markdown markers
page_count = llamaparse_markdown.count('\n## Page') + 1 if llamaparse_markdown else 0
if page_count == 0 and llamaparse_markdown: # Assume at least one page if there's content
    page_count = 1

# Create a ProcessingResult using the LlamaParse markdown
result = ProcessingResult(
    markdown=llamaparse_markdown,
    images=dummy_images,
    edges=dummy_edges,
    metadata={
        "source": PDF_PATH.name,
        "page_count": page_count,
        "_raw_marker_result": "Using LlamaParse instead of Marker"
    }
)

print("\nCreated ProcessingResult with markdown from LlamaParse.")


Success! LlamaParse generated 17,165 characters.

--- LlamaParse Markdown Preview (First 1000 chars) ---

HANS SHODH SUDHA, Vol. 1, Issue 4, (2021), pp. 64-70

# Applications of Mathematics in Machine Learning

Virender and Abhishek Pratap Singh

Department of Mathematics, Ramjas College, University of Delhi, Delhi. INDIA
Email :Virender57@yahoo.com; Abhishek.pr.singh03@gmail.com

> 

> **Abstract**
>
> 

> In this introductory report, use of mathematics in machine learning is studied and explained. Machine learning is an emerging area and now a day, it is one of essential components of technology in our day to day life. Through this study, an attempt is made to make an undergraduate student and a layman to understand mathematics involved in machine learning.

## 1. Introduction

Machine learning is basically a study of computer algorithms. It improves itself through experience and by the use of large data involved in experience. Artificial intelligence is the broad are involving mach

## 3. Build the Graph

Section hierarchy + figure/formula nodes + cross-references + spatial edges.

In [18]:
refs = extract_cross_references(result.markdown)
graph: nx.DiGraph = build_adjacency_map(result, refs=refs)

print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

type_counts = {}
for _, attr in graph.nodes(data=True):
    nt = attr.get("node_type", "unknown")
    type_counts[nt] = type_counts.get(nt, 0) + 1
pprint(type_counts)

Graph: 131 nodes, 130 edges
{'figure': 7, 'formula': 112, 'section': 5, 'text_block': 7}


## 4. Section Tree (LLM search target)

The LLM sees this tree with `[XXXX]` IDs and returns the ones relevant to the query.

`summary` is only available from LlamaParse's `get_tree(doc_id, node_summary=True)`; our local pipeline populates `text` instead.

In [ ]:
import json
from src.ingestion.tree import build_node_id_map

section_nodes = build_section_tree(graph)
node_id_map = build_node_id_map(graph)

print(f"📊 Top-level sections: {len(section_nodes)}")
print("\n🌲 Full tree:")
print(print_tree(section_nodes))

print("\n🌲 Raw tree (first node):")
if section_nodes:
    first = section_nodes[0]
    print(json.dumps({
        "title": first.title,
        "node_id": first.node_id,
        "page_index": first.page_index,
        "text": first.text[:500],
    }, indent=2))
else:
    print("(empty tree)")

print(f"\n🔗 node_id_map entries: {len(node_id_map)}")

## 5. Test with Mock LLM (no Ollama required)

Simulates tree search + answer synthesis. Replace with real Ollama below.

In [ ]:
from unittest.mock import MagicMock

QUERY = "What is the main contribution?"  # ← Change this

mock_llm = MagicMock()

# Mock step 1: LLM picks first 3 sections from the tree
if section_nodes:
    mock_ids = ",".join(n.node_id for n in section_nodes[:3])
    mock_llm.chat.return_value = mock_ids

    search_result = tree_search(QUERY, graph, mock_llm)
    print(f"LLM selected: {search_result.selected_ids}")
    print(f"Context: {len(search_result.context):,} chars")
    print()
    print("─ Context preview ─")
    print(search_result.context[:1500])
    print()

    # Mock step 2: answer
    mock_llm.chat.return_value = (
        f"Based on the sections [{', '.join(search_result.selected_ids[:3])}], "
        f"the main contribution is clearly explained in the extracted context."
    )
    answer = answer_query(QUERY, search_result.context, mock_llm)
    print("🧠 Answer:")
    print(answer)
else:
    print("No sections in the tree. The PDF may lack heading structure.")

No sections in the tree. The PDF may lack heading structure.


## 6. (Optional) Query with Real Ollama

If you have Ollama running and accessible (e.g., via ngrok or local network), configure below.

Otherwise skip — the mock test above already validates the pipeline.

In [ ]:
# ── Uncomment and configure if you have Ollama accessible ──

# OLLAMA_URL = "http://YOUR_SERVER_IP:11434"  # e.g. ngrok URL or local IP
# OLLAMA_MODEL = "llama3"

# llm = OllamaClient(base_url=OLLAMA_URL, model=OLLAMA_MODEL)

# if llm.is_available():
#     print("✅ Ollama connected")
#
#     QUERY = "Summarize the key findings of this paper."
#
#     search_result = tree_search(QUERY, graph, llm)
#     print(f"Selected: {search_result.selected_ids}")
#     print(f"Context: {len(search_result.context):,} chars")
#     print()
#
#     answer = answer_query(QUERY, search_result.context, llm)
#     print("🧠 Answer:")
#     print(answer)
# else:
#     print("❌ Ollama not reachable")